In [0]:
filepath="/Workspace/Repos/DataEngineerProject/PythonProject/Pyspark/apple_global_sales_dataset.csv"
df=spark.read.csv(filepath,header=True,inferSchema=True)
display(df)

In [0]:
from pyspark.sql import functions as F
df=df.filter(F.col("country")=="India")
display(df)

In [0]:
df=df.where(F.col("country")=="India")
display(df)

In [0]:
df.createOrReplaceTempView("apple_sales")



In [0]:
df=df.filter(F.col("country")=="India")
display(df)

In [0]:
df=df.filter(F.col("customer_rating").isNull())
display(df)

In [0]:
df=df.filter(F.col("customer_rating").isNotNull())
df.count()

In [0]:
#drop column
df=df.drop("quarter")
display(df)

In [0]:
df.distinct().count()

In [0]:
#dropping duplicates based on specified column
df=df.dropDuplicates(["category"])
display(df)

In [0]:
df=df.orderBy(F.col("country").desc())
display(df)


In [0]:
df=df.sort(F.col("sale_date").asc())
display(df)

In [0]:
from pyspark.sql import functions as F

df = (
    df.groupBy(
        F.month(F.col("sale_date")).alias("month_num"),
        F.col("month")
    )
    .agg(
        F.count("*").alias("count"),
        F.round(F.sum("revenue_usd"),2).alias("total_revenue")
    )
    .orderBy(F.col("month_num").asc())
)

df.show()

In [0]:
from pyspark.sql import functions as F

df = (
    df.groupBy(
        F.month("sale_date").alias("month")
    )
    .agg(
        F.count("*").alias("count"),
        F.round(F.sum(F.col("revenue_usd"))).alias("total_revenue")
    )
    .orderBy(F.col("month").asc())
)

display(df)
         


In [0]:
cus_df=spark.table("samples.tpch.customer")
display(cus_df)
order_df=spark.table("samples.tpch.orders")
display(order_df)

In [0]:
#join order df and cust df to get order information
join_df=(cus_df
         .select("c_custkey","c_name","c_phone")
         .join(order_df.select("o_orderkey","o_custkey"),
        on=cus_df.c_custkey==order_df.o_custkey,
        how="left"
        )
         .drop("o_custkey")
        .groupBy("c_custkey","c_name","c_phone")
        .agg(
            F.count(F.col("o_orderkey")).alias("total_orders")
            )
        )
    
display(join_df)

In [0]:
join_df=join_df.withColumn("is_vip_cus", F.when(F.col("total_orders") > 20,"True")
                           .otherwise("False"))
display(join_df)